In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# 1. Load the dataset
df = pd.read_csv('Groceries_dataset.csv')

# 2. Group items by Member_number and Date to define a "Transaction" (Basket)
# Each unique combination of Member and Date is considered one shopping trip.
transactions = df.groupby(['Member_number', 'Date'])['itemDescription'].apply(list).values.tolist()

# 3. One-Hot Encode the transactions
# The algorithm requires a True/False matrix for every item in every basket.
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

# 4. Apply the Apriori Algorithm
# min_support=0.001 means the itemset must appear in at least 0.1% of transactions.
frequent_itemsets = apriori(df_encoded, min_support=0.001, use_colnames=True)

# 5. Generate Association Rules
# We use 'lift' to find the strength of the association.
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)

# Sort rules by confidence to find the most reliable associations
rules = rules.sort_values('confidence', ascending=False)
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

                     antecedents   consequents   support  confidence      lift
235            (sausage, yogurt)  (whole milk)  0.001470    0.255814  1.619866
217        (sausage, rolls/buns)  (whole milk)  0.001136    0.212500  1.345594
228              (soda, sausage)  (whole milk)  0.001069    0.179775  1.138374
202        (semi-finished bread)  (whole milk)  0.001671    0.176056  1.114825
224         (yogurt, rolls/buns)  (whole milk)  0.001337    0.170940  1.082428
234        (sausage, whole milk)      (yogurt)  0.001470    0.164179  1.911760
112                  (detergent)  (whole milk)  0.001403    0.162791  1.030824
146                        (ham)  (whole milk)  0.002740    0.160156  1.014142
180           (processed cheese)  (rolls/buns)  0.001470    0.144737  1.315734
177  (packaged fruit/vegetables)  (rolls/buns)  0.001203    0.141732  1.288421
